## Dataset analysis: `fact_student_academic_performance_list16.csv`

Purpose: semester-level performance facts (List16). Useful as predictors and as consistency checks vs transcript (SEMESTER_GPA).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "fact_student_academic_performance_list16.csv")
df.shape

(34929, 15)

In [2]:
df.head()

,REG_NO,ACC_NO,PROGRAM,ACADEMIC_YEAR,SEMESTER,SEMESTER_INDEX,COURSES_REGISTERED,TOTAL_CREDITS,QUALITY_POINTS,PASSED_COURSES,FAILED_COURSES,FCW_COUNT,FEX_COUNT,MEX_COUNT,SEMESTER_GPA
0,AKM23B11/136,B21448,Bachelor of Laws,2024/2025,SEM1,1,7,21,86.0,6,1,1,0,0,4.10
1,AKM23B11/136,B21448,Bachelor of Laws,2024/2025,SEM2,2,6,18,69.0,5,1,1,0,0,3.83
2,AKM23B11/136,B21448,Bachelor of Laws,2025/2026,SEM1,3,7,21,100.0,7,0,0,0,0,4.76
3,AKM23B11/136,B21448,Bachelor of Laws,2025/2026,SEM2,4,6,17,75.0,6,0,0,0,0,4.41
4,AKM23B11/136,B21448,Bachelor of Laws,2026/2027,SEM1,5,7,21,95.5,7,0,0,0,0,4.55


In [3]:
df.isna().mean().sort_values(ascending=False).head(30)

REG_NO                0.0
ACC_NO                0.0
PROGRAM               0.0
ACADEMIC_YEAR         0.0
SEMESTER              0.0
SEMESTER_INDEX        0.0
COURSES_REGISTERED    0.0
TOTAL_CREDITS         0.0
QUALITY_POINTS        0.0
PASSED_COURSES        0.0
FAILED_COURSES        0.0
FCW_COUNT             0.0
FEX_COUNT             0.0
MEX_COUNT             0.0
SEMESTER_GPA          0.0
dtype: float64

In [4]:
key = ["REG_NO", "SEMESTER_INDEX"]
df.duplicated(key).sum(), df[key].isna().any(axis=1).sum()

(np.int64(0), np.int64(0))

In [5]:
num_cols = [c for c in df.columns if c not in ["REG_NO","ACC_NO","PROGRAM","ACADEMIC_YEAR","SEMESTER"]]
df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
SEMESTER_INDEX,34929.0,4.886169,3.258817,1.0,2.00,4.00,7.00,14.0
COURSES_REGISTERED,34929.0,6.534942,0.498785,6.0,6.00,7.00,7.00,7.0
TOTAL_CREDITS,34929.0,19.550374,1.653139,17.0,18.00,20.00,21.00,22.0
QUALITY_POINTS,34929.0,64.604483,14.995774,25.5,53.50,64.50,75.00,108.5
PASSED_COURSES,34929.0,5.184832,1.293513,0.0,4.00,5.00,6.00,7.0
FAILED_COURSES,34929.0,1.350110,1.234843,0.0,0.00,1.00,2.00,7.0
FCW_COUNT,34929.0,0.365685,0.588680,0.0,0.00,0.00,1.00,5.0
FEX_COUNT,34929.0,0.786624,1.040173,0.0,0.00,0.00,1.00,7.0
MEX_COUNT,34929.0,0.197801,0.437531,0.0,0.00,0.00,0.00,4.0
SEMESTER_GPA,34929.0,3.304704,0.714757,1.5,2.79,3.31,3.83,5.0


## Advanced analytics

Focus: semester performance components and how they track transcript CGPA / SEMESTER_GPA.

In [ ]:
from analysis_utils import basic_profile, missingness_report, merge_to_transcript_for_cgpa

print(basic_profile(df))
missingness_report(df, top_n=20)

BasicProfile(rows=34929, cols=15, dup_rows=0, null_cells=0)


,dtype,missing_rate,missing_count,nunique
REG_NO,str,0.0,0,5000
ACC_NO,str,0.0,0,5000
PROGRAM,str,0.0,0,3
ACADEMIC_YEAR,str,0.0,0,10
SEMESTER,str,0.0,0,2
SEMESTER_INDEX,int64,0.0,0,14
COURSES_REGISTERED,int64,0.0,0,2
TOTAL_CREDITS,int64,0.0,0,6
QUALITY_POINTS,float64,0.0,0,164
PASSED_COURSES,int64,0.0,0,8


In [ ]:
# Join to transcript to compare SEMESTER_GPA and components vs CGPA
trans = pd.read_csv(DATA_DIR / "student_transcript_list16.csv")
trans["CGPA"] = pd.to_numeric(trans["CGPA"], errors="coerce")

joined = merge_to_transcript_for_cgpa(df, trans, on=["REG_NO","SEMESTER_INDEX"], how="inner")

joined[[
    "CGPA",
    "SEMESTER_GPA",
    "COURSES_REGISTERED",
    "TOTAL_CREDITS",
    "PASSED_COURSES",
    "FAILED_COURSES",
    "FCW_COUNT",
    "FEX_COUNT",
    "MEX_COUNT",
]].corr(numeric_only=True)

,CGPA,SEMESTER_GPA,COURSES_REGISTERED,TOTAL_CREDITS,PASSED_COURSES,FAILED_COURSES,FCW_COUNT,FEX_COUNT,MEX_COUNT
CGPA,1.000000,0.860973,-0.002445,-0.001767,0.613355,-0.643485,-0.166610,-0.640871,-0.068352
SEMESTER_GPA,0.860973,1.000000,-0.001823,-0.003158,0.749428,-0.785771,-0.289331,-0.695832,-0.174147
COURSES_REGISTERED,-0.002445,-0.001823,1.000000,0.949897,0.307762,0.081541,0.048478,0.058876,0.024937
TOTAL_CREDITS,-0.001767,-0.003158,0.949897,1.000000,0.291581,0.078253,0.045779,0.056159,0.025747
PASSED_COURSES,0.613355,0.749428,0.307762,0.291581,1.000000,-0.923200,-0.407116,-0.754220,-0.264727
FAILED_COURSES,-0.643485,-0.785771,0.081541,0.078253,-0.923200,1.000000,0.446041,0.813836,0.287377
FCW_COUNT,-0.166610,-0.289331,0.048478,0.045779,-0.407116,0.446041,1.000000,-0.018588,-0.042408
FEX_COUNT,-0.640871,-0.695832,0.058876,0.056159,-0.754220,0.813836,-0.018588,1.000000,-0.055472
MEX_COUNT,-0.068352,-0.174147,0.024937,0.025747,-0.264727,0.287377,-0.042408,-0.055472,1.000000
